# 07 · Distribution Twin (Step 4)

The **Distribution / Fulfillment twin** — the customer-facing fulfillment network: which orders
are delivered, which customers are at risk, and how well we meet promised dates and quantities.

**Owns:** outbound deliveries, delivered-vs-ordered (fill rate), delivery delay, OTIF, partial
rate, fulfillment risk, ship-to context.

**Inputs:** `gold.outbounddeliveries` (rich pre-computed delivery facts).
**Outputs (monitoring):** `distribution_twin_current`, `distribution_twin_metrics_timeseries`.

Grain: material × plant × distribution channel (consistent with demand/forecast). _Missing for
a true distribution twin: shipments/transport + outlet network (see readiness scan)._ Idempotent overwrite.

## Config

In [ ]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("window_days", "365", "Recent window for current-state rollups")
dbutils.widgets.text("risk_otif_threshold", "0.90", "OTIF below this flags fulfillment risk")

import uuid
from pyspark.sql import functions as F

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
WINDOW = int(dbutils.widgets.get("window_days"))
RISK_OTIF = float(dbutils.widgets.get("risk_otif_threshold"))
REFRESH_ID = str(uuid.uuid4())

OD = f"{SRC}.outbounddeliveries"
AS_OF_SQL = f"(SELECT MAX(DeliveryCreationDate_ERDAT) FROM {OD})"
CUR_OUT = f"{FQ_MON}.distribution_twin_current"
TS_OUT  = f"{FQ_MON}.distribution_twin_metrics_timeseries"
print(f"Source: {OD}\nOutputs: {CUR_OUT} , {TS_OUT}\nWindow: {WINDOW}d")

## 1. Top ship-to region per series (context)
Largest-volume ship-to region per material × plant × channel (last 52 weeks).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW topregion_v AS
WITH c AS (
  SELECT MaterialNumber_MATNR AS material, Plant_WERKS AS plant, DistributionChannel_VTWEG AS channel,
         ShipToRegion_REGIO AS region, SUM(DeliveryQuantity_LFIMG) AS q
  FROM {OD}
  WHERE DeliveryCreationDate_ERDAT > date_sub({AS_OF_SQL}, 364) AND MaterialNumber_MATNR IS NOT NULL
  GROUP BY 1,2,3,4
),
r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY material,plant,channel ORDER BY q DESC) AS rn FROM c)
SELECT material, plant, channel, region AS top_ship_to_region FROM r WHERE rn = 1
""")
print("topregion_v ready")

## 2. Build `distribution_twin_current`
Recent-window fulfillment performance + open shipment book per material × plant × channel.

In [ ]:
current = spark.sql(f"""
SELECT b.*, tr.top_ship_to_region,
  (COALESCE(b.otif_rate, 0) < {RISK_OTIF}) AS fulfillment_risk_flag
FROM (
  SELECT
    Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
    DistributionChannel_VTWEG AS channel,
    MAX(MaterialText_MAKTX) AS material_text, MAX(MaterialGroup_MATKL) AS material_group, MAX(SalesUnit_VRKME) AS uom,
    COUNT(*) AS delivery_line_count,
    SUM(OrderedQuantity_KWMENG) AS ordered_qty, SUM(DeliveryQuantity_LFIMG) AS delivered_qty,
    CASE WHEN SUM(OrderedQuantity_KWMENG) > 0 THEN SUM(DeliveryQuantity_LFIMG) / SUM(OrderedQuantity_KWMENG) END AS fill_rate,
    AVG(CAST(OnTimeDelivery AS DOUBLE)) AS on_time_rate,
    AVG(CASE WHEN DeliveryQuantity_LFIMG >= OrderedQuantity_KWMENG THEN 1.0 ELSE 0.0 END) AS in_full_rate,
    AVG(CASE WHEN COALESCE(OnTimeDelivery, false) AND DeliveryQuantity_LFIMG >= OrderedQuantity_KWMENG
             THEN 1.0 ELSE 0.0 END) AS otif_rate,
    AVG(CAST(IsPartialDelivery AS DOUBLE)) AS partial_rate,
    AVG(CAST(DeliveryDelayDays AS DOUBLE)) AS avg_delay_days,
    AVG(CAST(DerivedTransitDays AS DOUBLE)) AS avg_transit_days,
    SUM(CASE WHEN NOT IsGoodsIssued THEN GREATEST(OrderedQuantity_KWMENG - DeliveryQuantity_LFIMG, 0) END) AS open_delivery_qty,
    CAST({AS_OF_SQL} AS DATE) AS as_of_date
  FROM {OD}
  WHERE MaterialNumber_MATNR IS NOT NULL
    AND DeliveryCreationDate_ERDAT > date_sub({AS_OF_SQL}, {WINDOW})
  GROUP BY 1,2,3,4
) b
LEFT JOIN topregion_v tr ON b.material=tr.material AND b.plant=tr.plant AND b.channel=tr.channel
""")
current = (current.withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(current.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CUR_OUT))
spark.sql(f"COMMENT ON TABLE {CUR_OUT} IS 'Distribution twin current state per material x plant x channel: fill rate, OTIF/on-time/in-full, delay, partial rate, open shipments, ship-to context, fulfillment risk.'")
print(f"Wrote {current.count():,} material-plant-channel rows to {CUR_OUT}")

## 3. Build `distribution_twin_metrics_timeseries`
Fulfillment performance over time (weekly by actual goods-issue date).

In [ ]:
timeseries = spark.sql(f"""
SELECT Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
       DistributionChannel_VTWEG AS channel,
       date_trunc('week', COALESCE(ActualGoodsIssueDate_WADAT_IST, DeliveryDate_LFDAT)) AS period_start,
  COUNT(*) AS delivery_line_count,
  SUM(OrderedQuantity_KWMENG) AS ordered_qty, SUM(DeliveryQuantity_LFIMG) AS delivered_qty,
  CASE WHEN SUM(OrderedQuantity_KWMENG) > 0 THEN SUM(DeliveryQuantity_LFIMG) / SUM(OrderedQuantity_KWMENG) END AS fill_rate,
  AVG(CAST(OnTimeDelivery AS DOUBLE)) AS on_time_rate,
  AVG(CASE WHEN COALESCE(OnTimeDelivery, false) AND DeliveryQuantity_LFIMG >= OrderedQuantity_KWMENG
           THEN 1.0 ELSE 0.0 END) AS otif_rate,
  AVG(CAST(DeliveryDelayDays AS DOUBLE)) AS avg_delay_days
FROM {OD}
WHERE MaterialNumber_MATNR IS NOT NULL
  AND COALESCE(ActualGoodsIssueDate_WADAT_IST, DeliveryDate_LFDAT) IS NOT NULL
GROUP BY 1,2,3,4,5
""")
timeseries = (timeseries.withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(timeseries.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TS_OUT))
spark.sql(f"COMMENT ON TABLE {TS_OUT} IS 'Distribution twin metric time-series per material x plant x channel x week: ordered/delivered qty, fill rate, on-time, OTIF, delay.'")
print(f"Wrote {timeseries.count():,} rows to {TS_OUT}")

## Column descriptions
Apply human-readable descriptions to every output column (persisted on rebuild).

In [ ]:
# Column descriptions — re-applied after the overwrite writes so they persist in the
# catalog on every rebuild. EXTEND for your business.
COLUMN_COMMENTS = {
    CUR_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "channel": "Distribution channel",
        "material_text": "Material description",
        "material_group": "Material group / category",
        "uom": "Sales unit of measure",
        "delivery_line_count": "Number of delivery lines in the window",
        "ordered_qty": "Total ordered quantity",
        "delivered_qty": "Total delivered quantity",
        "fill_rate": "Delivered divided by ordered quantity",
        "on_time_rate": "On-time delivery rate",
        "in_full_rate": "Share of lines delivered in full",
        "otif_rate": "On-time-in-full rate",
        "partial_rate": "Share of partial deliveries",
        "avg_delay_days": "Average delivery delay in days",
        "avg_transit_days": "Average transit time in days",
        "open_delivery_qty": "Open (not yet goods-issued) delivery quantity",
        "as_of_date": "As-of date of this twin snapshot",
        "top_ship_to_region": "Largest-volume ship-to region over the last 52 weeks",
        "fulfillment_risk_flag": "True when OTIF is below the configured threshold",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
    TS_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "channel": "Distribution channel",
        "period_start": "Start of the week bucket (actual goods-issue date)",
        "delivery_line_count": "Delivery lines in the week",
        "ordered_qty": "Ordered quantity in the week",
        "delivered_qty": "Delivered quantity in the week",
        "fill_rate": "Fill rate in the week",
        "on_time_rate": "On-time rate in the week",
        "otif_rate": "On-time-in-full rate in the week",
        "avg_delay_days": "Average delay in days in the week",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
}
for _tbl, _cols in COLUMN_COMMENTS.items():
    for _c, _desc in _cols.items():
        spark.sql(f"ALTER TABLE {_tbl} ALTER COLUMN {_c} COMMENT '{_desc}'")
print(f"Applied column descriptions to {CUR_OUT} and {TS_OUT}")

## 4. Verify

In [ ]:
print("=== distribution_twin_current (worst OTIF) ===")
display(spark.sql(f"""
SELECT material, plant, channel, material_group, delivery_line_count, ROUND(fill_rate,3) fill_rate,
       ROUND(on_time_rate,2) on_time, ROUND(otif_rate,2) otif, ROUND(avg_delay_days,1) delay_days,
       open_delivery_qty, top_ship_to_region, fulfillment_risk_flag
FROM {CUR_OUT} ORDER BY otif_rate ASC NULLS FIRST LIMIT 25
"""))
display(spark.sql(f"SELECT COUNT(*) rows, SUM(CAST(fulfillment_risk_flag AS INT)) at_risk, ROUND(AVG(fill_rate),3) avg_fill, ROUND(AVG(otif_rate),3) avg_otif FROM {CUR_OUT}"))
display(spark.sql(f"SELECT COUNT(*) rows, COUNT(DISTINCT material||plant||channel) series, MIN(period_start) first_p, MAX(period_start) last_p FROM {TS_OUT}"))

## Next
All five twins are live. Next stage: **Step 5 — KPI / control tower / twin views** (derive KPIs
from the twins; dashboards + Genie + direct twin drilldowns).